In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:

import torch
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} ГБ")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct", 
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True, 
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  
    random_state=42,
)
print("Модель загружена и готова к обучению")

In [ ]:
import csv, io, random
from datasets import Dataset
from collections import defaultdict


CANDIDATES_CSV = """Category,Title,Exp_Years,Key_Skills,Location,Salary_Min,Salary_Max,Employment,Remote,Summary
Фронтенд-разработчик,Frontend Developer,5,"HTML, CSS, JavaScript, React, Vue.js, Angular, Redux, Sass",Москва,80000,150000,Полная,Да,"Опыт создания адаптивных интерфейсов. Работа с современными фреймворками. Оптимизация производительности."
Фронтенд-разработчик,Senior Frontend Developer,6,"React, Vue.js, Angular, Redux, Vuex, Webpack, Babel",Санкт-Петербург,120000,200000,Полная,Гибрид,"Ведущий разработчик. Архитектура SPA. Управление состоянием приложения."
Фронтенд-разработчик,Frontend Developer,4,"HTML, CSS, JavaScript, React, Vue.js, Angular, Material-UI, Bootstrap",Москва,70000,130000,Полная,Да,"Разработка динамических приложений. UI/UX принципы. Адаптивный дизайн."
Фронтенд-разработчик,Frontend Developer,4,"HTML, CSS, JavaScript, React.js, Vue.js, Redux, Vuex, Git",Регионы,60000,110000,Полная,Удалённо,"Pixel-perfect вёрстка. State management. Контроль версий."
Фронтенд-разработчик,Frontend Developer,5,"HTML, CSS, JavaScript, React.js, Angular, Redux, REST API",Москва,85000,155000,Полная,Гибрид,"Интеграция с бэкендом. Управление состоянием. Кроссбраузерность."
Фронтенд-разработчик,Frontend Developer,4,"HTML, CSS, JavaScript, React.js, Vue.js, Svelte, Styled-Components",Санкт-Петербург,75000,140000,Полная,Да,"Современные фреймворки. CSS-in-JS. Адаптивность."
Бэкенд-разработчик,Backend Developer,7,"Node.js, Express.js, MongoDB, PostgreSQL, MySQL, AWS, Azure",Москва,100000,180000,Полная,Да,"RESTful API. Микросервисы. Облачные технологии."
Бэкенд-разработчик,Senior Backend Developer,8,"Node.js, Express.js, MongoDB, PostgreSQL, JWT, OAuth, AWS",Москва,150000,250000,Полная,Гибрид,"Архитектура серверных решений. Аутентификация. Масштабирование."
Бэкенд-разработчик,Backend Developer,5,"Node.js, Flask, Django, Express.js, MySQL, PostgreSQL, Redis, Kafka",Санкт-Петербург,90000,160000,Полная,Да,"REST API. Message brokers. Контейнеризация."
"""

VACANCIES_CSV = """Category,Title,Exp_Years,Key_Skills,Location,Salary_Min,Salary_Max,Employment,Remote,Summary
Фронтенд-разработчик,Frontend Developer,3-5,"React, Vue, Angular, HTML/CSS, JavaScript, Redux",Москва,80000,150000,Полная,Да,"Разработка пользовательских интерфейсов для веб-приложений. Работа с современными фронтенд-фреймворками."
Фронтенд-разработчик,Senior Frontend Developer,5-7,"React, TypeScript, Webpack, CSS-in-JS, GraphQL",Санкт-Петербург,120000,200000,Полная,Гибрид,"Ведущий разработчик фронтенда. Архитектура SPA, оптимизация производительности."
Фронтенд-разработчик,Frontend Developer (Vue),3-5,"Vue.js, Vuex, Nuxt, JavaScript, HTML5, CSS3",Москва,90000,160000,Полная,Да,"Разработка на Vue.js. Создание адаптивных интерфейсов, работа с API."
Фронтенд-разработчик,Frontend Developer (Angular),4-6,"Angular, TypeScript, RxJS, NgRx, Material",Москва,100000,170000,Полная,Гибрид,"Разработка корпоративных приложений на Angular. Работа в команде."
Фронтенд-разработчик,Junior Frontend Developer,1-2,"HTML, CSS, JavaScript, React, Git",Регионы,50000,80000,Полная,Да,"Начальная позиция для фронтенд-разработчика. Обучение и наставничество."
Фронтенд-разработчик,Frontend Team Lead,6-8,"React, TypeScript, Team Management, Architecture",Москва,180000,280000,Полная,Гибрид,"Руководство командой фронтенд-разработчиков. Архитектурные решения."
Бэкенд-разработчик,Backend Developer,3-6,"Node.js, Express, Python, Django, PostgreSQL, MongoDB",Москва,90000,160000,Полная,Да,"Разработка серверной логики и API. Работа с базами данных."
Бэкенд-разработчик,Senior Backend Engineer,6-8,"Java, Spring Boot, Microservices, Kafka, Docker, AWS",Москва,150000,250000,Полная,Нет,"Ведущий бэкенд-разработчик. Микросервисная архитектура, высоконагруженные системы."
Бэкенд-разработчик,Backend Developer (Python),3-5,"Python, Django, Flask, PostgreSQL, Redis",Регионы,70000,130000,Полная,Да,"Разработка на Python. REST API, интеграции, работа с БД."
"""

def parse_csv(text):
    return list(csv.DictReader(io.StringIO(text.strip())))

cands = parse_csv(CANDIDATES_CSV)
vacs = parse_csv(VACANCIES_CSV)

c_by_cat = defaultdict(list)
v_by_cat = defaultdict(list)
for c in cands: c_by_cat[c['Category']].append(c)
for v in vacs: v_by_cat[v['Category']].append(v)

def parse_exp(s):
    s = s.strip().replace('+', '')
    if '-' in s: return int(s.split('-')[0])
    return int(s) if s.isdigit() else 1

pairs = []
for cat in c_by_cat:
    if cat not in v_by_cat: continue
    for c in c_by_cat[cat]:
        v = random.choice(v_by_cat[cat])

        c_skills = set(c['Key_Skills'].lower().replace(', ', ',').split(','))
        v_skills = set(v['Key_Skills'].lower().replace(', ', ',').split(','))
        skill_match = len(c_skills & v_skills) / len(c_skills | v_skills) if len(c_skills | v_skills) > 0 else 0

        c_exp = parse_exp(c['Exp_Years'])
        v_exp_min = parse_exp(v['Exp_Years'])
        exp_match = 1.0 if c_exp >= v_exp_min else max(0.5, c_exp / v_exp_min)

        c_sal = (int(c['Salary_Min']) + int(c['Salary_Max'])) / 2
        v_sal = (int(v['Salary_Min']) + int(v['Salary_Max'])) / 2
        sal_match = 1.0 if c_sal <= v_sal * 1.1 else 0.7

        score = int((skill_match * 0.5 + exp_match * 0.3 + sal_match * 0.2) * 100)
        score = max(20, min(95, score))

        reason = []
        if skill_match > 0.6: reason.append("Соответствует требованиям по навыкам")
        elif skill_match > 0.3: reason.append("Частичное совпадение стека")
        else: reason.append("Ключевые технологии отсутствуют")

        if c_exp < v_exp_min: reason.append("Недостаточно опыта")
        if c['Remote'] != v['Remote'] and v['Remote'] != 'Гибрид': reason.append("Не совпадает формат работы")

        prompt = (
            f"Оцени соответствие кандидата вакансии.\n"
            f"Резюме: {c['Summary']}\nСтаж: {c['Exp_Years']} лет\nНавыки: {c['Key_Skills']}\n"
            f"Зарплата: {c['Salary_Min']}-{c['Salary_Max']} руб.\n\n"
            f"Вакансия: {v['Summary']}\nТребуемый стаж: {v['Exp_Years']} лет\n"
            f"Навыки: {v['Key_Skills']}\nЗарплата: {v['Salary_Min']}-{v['Salary_Max']} руб.\nФормат: {v['Remote']}"
        )
        formatted_sample = (
            f"<|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n{response}<|eot_id|>"
        )
        pairs.append(formatted_sample)

random.seed(42)
random.shuffle(pairs)
dataset = Dataset.from_dict({"text": pairs[:70]})  
print(f"✅ Датасет готов: {len(dataset)} пар | Пример:\n{dataset['text'][0][:200]}...")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq

num_samples = len(dataset)
effective_batch = 2 * 4  
steps_per_epoch = max(1, num_samples // effective_batch)
max_steps = max(5, steps_per_epoch * 4)  
warmup = max(1, int(max_steps * 0.2))    

print(f"Датасет: {num_samples} примеров | Эффективный батч: {effective_batch} | Шагов: {max_steps}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,  
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=warmup,
        max_steps=max_steps,          
        learning_rate=1e-4,           
        fp16=True,                   
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        logging_steps=1,              
        save_strategy="no",
        output_dir="outputs",
        seed=42,
    ),
)

print("Запуск обучения...")
trainer.train()
print("Обучение завершено!")

In [ ]:

model.save_pretrained_gguf(
    "screening-test",
    tokenizer,
    quantization_method="q4_k_m",  
)

!curl -fsSL https://ollama.com/install.sh | sh
!ollama serve &
!sleep 5
!ollama create screening-test:3b -f ./screening-test/Modelfile

print("Модель зарегистрирована в Ollama")
print("Скачать папку 'screening-test/' через файловый менеджер Colab (слева → ПКМ → Download)")

In [ ]:
import shutil
import os

# Создаём ZIP-архив
shutil.make_archive('/content/screening-test', 'zip', '/content/screening-test')
print("✅ Архив создан: /content/screening-test.zip")

# Скачиваем
from google.colab import files
files.download('/content/screening-test.zip')
print("Скачивание началось...")

In [ ]:
import os
import shutil
from google.colab import files

# 1. Создаем папку для экспорта
output_dir = "ollama_export"
os.makedirs(output_dir, exist_ok=True)

print("1. Конвертация в формат Ollama (GGUF)...")
model.save_pretrained_gguf(output_dir, tokenizer, quantization_method="q4_k_m")

gguf_file = None
for f in os.listdir(output_dir):
    if f.endswith(".gguf"):
        gguf_file = f
        break

if gguf_file:
    print(f"GGUF найден: {gguf_file}")

    modelfile_content = f"""
FROM ./{gguf_file}
PARAMETER temperature 0.1
PARAMETER top_p 0.9
"""
    with open(os.path.join(output_dir, "Modelfile"), "w", encoding="utf-8") as f:
        f.write(modelfile_content)

    final_pack_dir = "final_model_pack"
    os.makedirs(final_pack_dir, exist_ok=True)

    shutil.copy(os.path.join(output_dir, gguf_file), final_pack_dir)
    shutil.copy(os.path.join(output_dir, "Modelfile"), final_pack_dir)

    # 5. Упаковываем и скачиваем
    zip_name = "model_for_ollama"
    shutil.make_archive(zip_name, "zip", final_pack_dir)

    print("5. Скачиваем архив...")
    files.download(f"{zip_name}.zip")
    print("Готово! Жди завершения загрузки.")
else:
    print("Ошибка: GGUF файл не найден. Возможно, модель уже выгружена из памяти.")

In [ ]:
from google.colab import files
files.download('/content/screening-test.zip')
print("Скачивание началось...")